# PROJECT 5


In [ ]:
from transformers import AutoTokenizer
import torch

class TextProcessor:
    def __init__(self, model_name="distilbert-base-uncased"):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)

    def prepare_input(self, text: str):
        if not text.strip():
            raise ValueError("Input text cannot be empty.")

        return self.tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            padding="max_length",
            max_length=128
        )

In [ ]:
import logging
import time
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from transformers import AutoModelForSequenceClassification
import torch

# Setup Logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("nlp-service")

app = FastAPI(title="NLP Sentiment Service")
processor = TextProcessor()
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english")

class SentimentRequest(BaseModel):
    text: str

@app.post("/predict")
async def predict(request: SentimentRequest):
    start_time = time.time()
    try:
        inputs = processor.prepare_input(request.text)

        with torch.no_grad():
            outputs = model(**inputs)
            probabilities = torch.nn.functional.softmax(outputs.logits, dim=-1)
            prediction = torch.argmax(probabilities).item()
            confidence = torch.max(probabilities).item()

        latency = time.time() - start_time
        logger.info(f"Prediction successful. Latency: {latency:.4f}s")

        return {
            "label": "POSITIVE" if prediction == 1 else "NEGATIVE",
            "confidence": round(confidence, 4),
            "latency": f"{latency:.4f}s"
        }
    except Exception as e:
        logger.error(f"Prediction failed: {str(e)}")
        raise HTTPException(status_code=500, detail="Internal Model Error")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [ ]:
from fastapi.testclient import TestClient

client = TestClient(app)

def test_predict_positive():
    response = client.post("/predict", json={"text": "I love this product, it works great!"})
    assert response.status_code == 200
    assert response.json()["label"] == "POSITIVE"

def test_empty_input():
    response = client.post("/predict", json={"text": ""})
    # This should trigger our 500 or a custom 400 if implemented
    assert response.status_code == 500